# Day 40: Integrate Conversational Memory

Welcome to Day 40! Today, we are focusing on a critical component of building Agentic AI: **Conversational Memory**.

## Core Theory (Just-in-Time)

**Why Memory?**
By default, Large Language Models (LLMs) and standard LLM chains are **stateless**. They do not remember previous interactions. If you ask an LLM "What is my name?" and then in the next request ask "What did I just ask you?", it will have no idea.
In production, for chatbots or agents to be useful, they need context about the ongoing conversation.

**How does Memory work?**
At a high level, memory works by intercepting the user's input, appending the history of previous messages to the prompt, and sending the combined prompt to the LLM. The LLM's response is then saved back into the memory for the next turn.

**Types of Memory in LangChain:**
1.  **`ConversationBufferMemory`**: Stores the raw, complete transcript of the conversation. *Pros:* Perfect recall. *Cons:* Context window fills up quickly, leading to higher token costs and potential "Lost in the Middle" issues.
2.  **`ConversationBufferWindowMemory`**: Keeps a sliding window of the last *k* interactions. *Pros:* Bounded token usage. *Cons:* Forgets older details (like the user's name mentioned at the start).
3.  **`ConversationSummaryMemory`**: Uses an LLM to dynamically summarize the conversation as it happens. *Pros:* Long-term context with small token footprint. *Cons:* Requires extra LLM calls (latency/cost) and might lose fine-grained details during summarization.

Today, we'll focus on modern LCEL (LangChain Expression Language) implementations for memory, specifically `RunnableWithMessageHistory`, alongside buffer concepts and summarization concepts.

## Code Implementation

Let's look at how to implement conversational memory using the modern `RunnableWithMessageHistory` abstraction in LangChain.

In [1]:
import os
from typing import Dict, Any, List
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory

# 1. Setup global store for chat histories
# In production, this would be a database like Redis or Postgres.
store: Dict[str, BaseChatMessageHistory] = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    """
    Retrieves or creates a chat message history for a given session.
    
    Args:
        session_id (str): The unique identifier for the user session.
        
    Returns:
        BaseChatMessageHistory: The chat history object for the session.
    """
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


### Example 1: RunnableWithMessageHistory (Window / Buffer)

This is the recommended stable pattern for adding memory to an LCEL chain. It automatically manages injecting historical messages and saving new ones.

In [2]:
def run_conversation_example() -> None:
    """
    Demonstrates setting up a conversational chain using LCEL and RunnableWithMessageHistory.
    """
    print("--- Conversational Memory Example ---")
    
    try:
        # 1. Initialize the LLM
        llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
        
        # 2. Define the Prompt
        # MessagesPlaceholder tells LangChain where to inject the chat history
        prompt = ChatPromptTemplate.from_messages([
            ("system", "You are a helpful assistant."),
            MessagesPlaceholder(variable_name="history"),
            ("human", "{question}"),
        ])
        
        # 3. Create the basic chain
        chain = prompt | llm
        
        # 4. Wrap the chain with Message History
        with_message_history = RunnableWithMessageHistory(
            chain,
            get_session_history,
            input_messages_key="question",
            history_messages_key="history",
        )
        
        # Simulate a conversation
        session_config = {"configurable": {"session_id": "user_123"}}
        
        res1 = with_message_history.invoke(
            {"question": "Hi, my name is Bob."},
            config=session_config
        )
        print(f"AI: {res1.content}")
        
        res2 = with_message_history.invoke(
            {"question": "What is my name?"},
            config=session_config
        )
        print(f"AI: {res2.content}")
        
        # Inspecting the saved history
        print(f"\nCurrent Memory State for user_123:\n{store['user_123'].messages}")
        
    except Exception as e:
        print(f"Execution requires a valid OpenAI API key. Error: {e}")

run_conversation_example()


--- Conversational Memory Example ---


Parent run e3211791-947d-46da-a5a6-30c87616e424 not found for run b7d41650-78bd-430a-b419-d0833183da6f. Treating as a root run.


Execution requires a valid OpenAI API key. Error: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-dummy-key. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}


### Example 2: ConversationSummaryMemory

While `RunnableWithMessageHistory` is excellent for tracking raw messages, sometimes we need to compress a long chat history to save tokens. LangChain's `ConversationSummaryMemory` leverages an LLM to build a running summary.

In [3]:
from langchain.memory import ConversationSummaryMemory
from langchain.chains import LLMChain
from langchain_core.prompts import PromptTemplate

def run_summary_memory_example() -> None:
    """
    Demonstrates how to use ConversationSummaryMemory to compress chat history using an LLM.
    """
    print("\n--- Summary Memory Example ---")
    
    try:
        # 1. Initialize the Summarization LLM
        # Typically we use a cheap, fast model for the summarizer.
        summary_llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
        
        # 2. Initialize the Summary Memory object
        memory = ConversationSummaryMemory(
            llm=summary_llm, 
            memory_key="chat_history",
            return_messages=False
        )
        
        # We can manually save context to see the summarizer in action.
        # This will trigger an LLM call under the hood to update the summary!
        print("Saving context 1...")
        memory.save_context(
            {"input": "Hi, I'm building an AI agent."}, 
            {"output": "That sounds exciting! What stack are you using?"}
        )
        
        print("Saving context 2...")
        memory.save_context(
            {"input": "I'm using Python, LangChain, and Qdrant."}, 
            {"output": "Great choices. Qdrant is excellent for vector storage."}
        )
        
        # 3. Retrieve the generated summary
        memory_vars = memory.load_memory_variables({})
        print(f"\nGenerated Conversation Summary:\n{memory_vars['chat_history']}")
        
    except Exception as e:
        print(f"Summary memory requires a valid OpenAI API key to run. Error: {e}")

run_summary_memory_example()



--- Summary Memory Example ---
Saving context 1...


Summary memory requires a valid OpenAI API key to run. Error: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-dummy-key. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}


## Common Pitfalls in Production

1.  **Memory Leakage (Token Limits):** A long chat session will quickly exceed the context window. When using `ChatMessageHistory` in production, you must often combine it with trimming techniques (like `trim_messages` in LangChain) to simulate window memory.
2.  **Stateless API Design:** If you are building an API (e.g., FastAPI), the memory object cannot be instantiated globally in Python memory (like our `store` dict). Every user/session needs its own memory instance retrieved from a database (like Redis) at the start of the request.
3.  **Summarization Latency:** Summarizing conversations requires additional LLM calls (`ConversationSummaryMemory` blocks execution during summarization). If your summarization LLM is slow, the user waits longer for their response. Consider doing summarization asynchronously.
4.  **Losing Fine Details:** If you summarize too aggressively, the model might forget specific names, IDs, or error codes mentioned earlier. A hybrid approach (Window + Summary) is often best.

## Practical Lab

**Your Task:**
Build a windowed memory buffer locally!
1. Initialize a `ConversationBufferWindowMemory` from `langchain.memory`.
2. Configure it with `k=3` (to retain only the last 3 interactions).
3. Insert 5 mock interactions using `save_context`.
4. Print the final buffer state to verify it strictly bounded the memory.

In [4]:
from langchain.memory import ConversationBufferWindowMemory

def run_lab_implementation() -> None:
    """
    Executes the practical lab: setting up a bounded window memory and verifying it truncates older messages.
    """
    print("\n--- Lab Execution ---")
    
    # 1. Initialize Window Memory (k=3)
    lab_memory = ConversationBufferWindowMemory(k=3, memory_key="chat_history")
    
    # 2. Define 5 interactions
    interactions = [
        ("Hello", "Hi there"),
        ("My name is Charlie", "Nice to meet you, Charlie"),
        ("I like Python", "Python is great"),
        ("What's 2+2?", "4"),
        ("What did I say my name was?", "You said your name was Charlie")
    ]
    
    # 3. Save contexts
    for user_msg, ai_msg in interactions:
        lab_memory.save_context({"input": user_msg}, {"output": ai_msg})
    
    # 4. Print final buffer state
    print("Final Buffer State (Should only contain the last 3 interactions):")
    print(lab_memory.buffer)

run_lab_implementation()



--- Lab Execution ---
Final Buffer State (Should only contain the last 3 interactions):
Human: I like Python
AI: Python is great
Human: What's 2+2?
AI: 4
Human: What did I say my name was?
AI: You said your name was Charlie
